In [ ]:
import re
import json
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
# 假設你已經完成了 driver 的初始化，例如：
# driver = webdriver.Chrome()
# driver.get("你的購物網站首頁URL")

# ⚠️ 1. 設置一個「總體」列表來存放所有頁面的商品資訊
all_product_info = []

page_number = 1
while True:
    print(f"--- 開始處理第 {page_number} 頁 ---")

    # ----------------------------------------------------
    # A. 核心：單頁商品資訊抓取 (把你的第一個程式碼塊放進來)
    # ----------------------------------------------------
    try:
        # 獲得 script 中的文字 (這部分通常是固定的)
        # 為了避免在後續頁面找不到元素時報錯，可以加一個 try-except
        data_element = driver.find_element(By.XPATH, "/html/body/script[@type='application/ld+json']")
        data_string = data_element.get_attribute('innerHTML')

        # 正則表達式清除裡面的錯誤, 獲得 dict
        # 注意：這個 fix_data_string = re.sub(r",\s*]", "]", data_string) 
        # 是針對特定網站的資料格式修正，請確保在所有頁面都適用。
        fix_data_string = re.sub(r",\s*]", "]", data_string)
        data_json = json.loads(fix_data_string)

        # 從 dict 中，找到包含商品資訊的 list
        product_list = data_json['mainEntity']['itemListElement']
        
        # 從商品 list 中再找出想要的資訊，並加入總體列表
        current_page_products = [] # 儲存當前頁面的商品，可選
        for product in product_list:
            # 增加 try-except 處理個別商品資料欄位缺失 (提高健壯性)
            try:
                product_name = product.get('name', 'N/A') # 使用 .get() 避免 Key Error
                product_price = product.get('offers', {}).get('price', 'N/A')
                product_image = product.get('image', 'N/A')
                product_link = product.get('url', 'N/A')
                
                info_dict = {
                    "product_name" : product_name,
                    "product_price(NTD)": product_price,
                    "product_image": product_image,
                    "product_url": product_link,
                    "source_page": page_number # 記錄來源頁碼，有利於後續除錯和分析
                }

                all_product_info.append(info_dict)
                current_page_products.append(info_dict) # 存入當前頁面列表

            except Exception as e:
                print(f"⚠️ 警告：處理第 {page_number} 頁中的單一商品時發生錯誤: {e}")
                continue # 跳過這個有問題的商品

        print(f"✅ 第 {page_number} 頁成功抓取 {len(current_page_products)} 筆資料。")
        
    except Exception as e:
        print(f"❌ 警告：抓取或解析第 {page_number} 頁的 JSON 資料時發生錯誤: {e}")
        # 如果第一頁就失敗，可能是定位錯誤；如果是中間頁失敗，可能網頁結構變了，此處可以選擇 break 或 continue
        # 我們這裡選擇繼續檢查是否有下一頁按鈕，以便流程不中斷。
        pass # 繼續執行 B 區塊

    # ----------------------------------------------------
    # B. 分頁點擊邏輯 (你的第二個程式碼塊)
    # ----------------------------------------------------
    
    # 查找「下一頁」按鈕
    # 這裡使用 find_elements 是正確的，因為當元素不存在時它會返回空列表 [] 而不是報錯
    next_page_btns = driver.find_elements(By.CSS_SELECTOR, ".page-btn.page-next")
    
    if next_page_btns:
        # 找到按鈕，進行翻頁
        next_page_btn = next_page_btns[0]
        next_page_btn.click()
        
        # ⚠️ **關鍵優化：等待機制**
        # 由於網站內容是動態載入的，建議使用更智慧的等待方式
        # 這裡我們保留 time.sleep(3)，但未來建議替換為 WebDriverWait + expected_conditions 
        time.sleep(3) # 給予足夠時間讓網頁載入下一頁的內容
        
        page_number += 1
    else:
        # 沒有找到下一頁按鈕，表示已到達最後一頁
        print("--- 🏁 找不到下一頁按鈕，爬取結束。 ---")
        break

# ----------------------------------------------------
# C. 輸出結果
# ----------------------------------------------------
print("\n--- 總結 ---")
print(f"總共抓取了 {page_number} 頁資料。")
print(f"商品總數：{len(all_product_info)} 筆。")
# 輸出所有抓取到的資料 (可以考慮存成 CSV 或 JSON 檔案)
# print(all_product_info)

In [9]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import json
import time
# 建立 Service 物件，指定 chromedriver.exe 的路徑

# 設定 Chrome 瀏覽器的選項
options = webdriver.ChromeOptions()
options.add_argument("--start-maximized") # Chrome 瀏覽器在啟動時最大化視窗 避免 rwd
options.add_argument("--incognito") # 無痕模式 
options.add_argument("--disable-popup-blocking") # 停用 Chrome 的彈窗阻擋功能。

# 建立 Chrome 瀏覽器物件
driver = webdriver.Chrome(options=options)
driver.get("https://24h.pchome.com.tw/")
time.sleep(2)

# 關閉廣告
ad_close_btn = driver.find_element(By.CSS_SELECTOR, ".c-popUp__endBtn button")
if ad_close_btn:
    try:
        ad_close_btn.click()
    except Exception as e:
        print("沒有廣告")

# 進入搜尋,先清除內容

insert_product = driver.find_element(By.CSS_SELECTOR, ".c-search__search input").clear()
insert_product = driver.find_element(By.CSS_SELECTOR, ".c-search__search input").send_keys("Python")
start_find = driver.find_element(By.CSS_SELECTOR, ".c-search__btn.c-search__btn--search button").click()




In [10]:
# 進入商品列
product_elements = driver.find_elements(By.CSS_SELECTOR, ".c-listInfoGrid__body ul li")
print(f"找到{len(product_elements)}個商品")
# product_data_list = []
# for product in product_elements:
#     product_name = product.find_element(By.CSS_SELECTOR, ".listAreaLi .goodsUrl .prdNameTitle").text
#     product_price = product.find_element(By.CSS_SELECTOR, ".listAreaLi .goodsUrl .prdInfoWrap .money").text
#     product_image = product.find_element(By.CSS_SELECTOR, ".listAreaLi .goodsUrl .prdImgWrap .swiper .swiper-wrapper .swiper-slide .goods-img-url picture img").get_attribute("src")
#     # print(product_image)
#     product_link = product.find_element(By.TAG_NAME, "a").get_attribute("href")
#     print(product_link)

#     # product_data_list.append({
#     #     print(f"名稱: {product_name}")
#     #     print (f"")
#     #     print (f"")
#     # })
#     break


找到203個商品


In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import json
import time
# 建立 Service 物件，指定 chromedriver.exe 的路徑

# 設定 Chrome 瀏覽器的選項
options = webdriver.ChromeOptions()
options.add_argument("--start-maximized") # Chrome 瀏覽器在啟動時最大化視窗 避免 rwd
options.add_argument("--incognito") # 無痕模式 
options.add_argument("--disable-popup-blocking") # 停用 Chrome 的彈窗阻擋功能。

# 建立 Chrome 瀏覽器物件
driver = webdriver.Chrome(options=options)
driver.get("https://www.momoshop.com.tw/")

time.sleep(5)


In [3]:
# 清空搜尋欄並輸入商品名按搜尋

cancle_product = driver.find_element(By.CSS_SELECTOR, ".relative.flex-1 input").clear()
insert_product = driver.find_element(By.CSS_SELECTOR, ".relative.flex-1 input").send_keys("衛生紙")
start_find = driver.find_element(By.CSS_SELECTOR, ".flex.h-full.items-center.overflow-hidden button").click()

In [37]:
# 取得 script 內的 json 格式元素

try:
    data_element = driver.find_element(By.CSS_SELECTOR, "script[type='application/ld+json']")
except Exception as e:
    print("查無此項商品")

data_string = driver.execute_script("return arguments[0].textContent;",data_element)
json_data_elements = json.loads(data_string)
# print(json_data_elements)
# # print(type(json_data_elements))
# # 成功獲得 dict


product_list = json_data_elements.get('mainEntity',{}).get("itemListElement", [])
print(product_list)
# 獲得商品list

[{'@type': 'Product', 'name': '【YAMAZAKI】tower沉蓋式面紙盒-白(面紙盒/抽取式面紙盒/面紙盒/衛生紙盒)', 'position': '1', 'image': 'https://img3.momoshop.com.tw/goodsimg/0009/656/372/9656372_OL.jpg?t=1693158374', 'description': '日本百年品牌 總代理公司貨', 'url': 'https://www.momoshop.com.tw/goods/GoodsDetail.jsp?i_code=9656372&Area=search&mdiv=403&oid=1_1&cid=index&kw=%E8%A1%9B%E7%94%9F%E7%B4%99', 'offers': {'@type': 'Offer', 'price': '1145', 'priceCurrency': 'TWD', 'availability': 'https://schema.org/InStock'}}, {'@type': 'Product', 'name': '【Vinda 維達】線條小狗聯名款 掛式3層衛生紙 280抽*1包 尺寸14.6*19.5cm 加厚立體壓花 100%原生木漿 抽取衛生紙', 'position': '2', 'image': 'https://i6.momoshop.com.tw/1763185950/goodsimg/TP000/8488/0000/008/TP00084880000008_O_m.jpg', 'description': '小羊姐獨家維達x線條小狗聯名', 'url': ' https://www.momoshop.com.tw/TP/TP0008488/goodsDetail/TP00084880000008', 'offers': {'@type': 'Offer', 'price': '$89', 'priceCurrency': 'TWD', 'availability': 'https://schema.org/InStock'}, 'aggregateRating': {'@type': 'AggregateRating', 'ratingValue': 0, 

In [ ]:
# 將我們需要的商品名稱、價格、圖片、購買連結取出
product_data = []
for item in product_list:
    if item.get('@type') == 'Product' and 'offers' in item:
        offers = item.get('offers', {})
        product_info = {
            "商品名": item.get('name', 'N/A'),
            "價格(新台幣)": offers.get('price', 'N/A'), 
            "圖片URL": item.get('image', 'N/A'),
            "購買連結": item.get('url', 'N/A')
        }
        product_data.append(product_info)
    else:
        print("商品資訊不全，不顯示")
# for p in product_data[:5]:
#     print(p)
print(product_data)

[{'商品名': '【MAY FLOWER 五月花】厚棒抽取式衛生紙(90抽x60包)', '價格': '1169', '圖片URL': 'https://img4.momoshop.com.tw/goodsimg/0006/532/205/6532205_OL.jpg?t=1763085631', '購買連結': 'https://www.momoshop.com.tw/goods/GoodsDetail.jsp?i_code=6532205&Area=search&mdiv=403&oid=1_1&cid=index&kw=%E8%A1%9B%E7%94%9F%E7%B4%99'}, {'商品名': '【YAMAZAKI】tower沉蓋式面紙盒-白(面紙盒/抽取式面紙盒/面紙盒/衛生紙盒)', '價格': '1145', '圖片URL': 'https://img4.momoshop.com.tw/goodsimg/0009/656/372/9656372_OL.jpg?t=1693158374', '購買連結': 'https://www.momoshop.com.tw/goods/GoodsDetail.jsp?i_code=9656372&Area=search&mdiv=403&oid=1_2&cid=index&kw=%E8%A1%9B%E7%94%9F%E7%B4%99'}, {'商品名': '【Vinda 維達】線條小狗聯名款 掛式3層衛生紙 280抽*1包 尺寸14.6*19.5cm 加厚立體壓花 100%原生木漿 抽取衛生紙', '價格': '$89', '圖片URL': 'https://i8.momoshop.com.tw/1763185950/goodsimg/TP000/8488/0000/008/TP00084880000008_O_m.jpg', '購買連結': ' https://www.momoshop.com.tw/TP/TP0008488/goodsDetail/TP00084880000008'}, {'商品名': '【倍潔雅】花漾柔感抽取式衛生紙PEFC(150抽x84包/箱)', '價格': '979', '圖片URL': 'https://img4.momoshop.com.tw/goodsimg/0008/037/

In [ ]:
# 找下一頁按鈕,並點案持續搜尋

page_number = 1

while True:
    next_page_btn= driver.find_elements(By.CSS_SELECTOR, ".page-btn.page-next")
    if next_page_btn:
        next_page_btn = next_page_btn[0]
        next_page_btn.click()
        time.sleep(3)
        page_number += 1
    else:
        break